<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/04-spark-sql/00-temp-views-and-spark-sql.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 4.0 — Temp views, global temp views, spark.sql()

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("4.0-temp-views")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## spark.sql() returns an ordinary, lazy DataFrame

In [ ]:
df = spark.sql("SELECT 1 AS one, 'hello' AS greeting")
print(type(df))
df.show()

## Register a view, query it — and prove no data was copied

In [ ]:
orders.createOrReplaceTempView("orders")

top_countries = spark.sql("""
    SELECT country, ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM orders
    WHERE country IS NOT NULL
    GROUP BY country
    ORDER BY revenue DESC
""")
top_countries.show()

In [ ]:
# Analysis is eager (typos fail NOW), execution is lazy (no job until action)
try:
    spark.sql("SELECT quantiy FROM orders")
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## Round trip: SQL -> DataFrame ops -> view -> SQL — one fused plan

In [ ]:
enriched = top_countries.withColumn("revenue_eur", F.round(col("revenue") * 0.92, 2))
enriched.createOrReplaceTempView("revenue_eur")
final = spark.sql("SELECT * FROM revenue_eur WHERE revenue_eur > 100")
final.show()
final.explain()   # no seams where the languages switched

## The stale-view bug, reproduced

In [ ]:
orders = orders.filter(col("category") == "books")     # redefine the VARIABLE
print("python variable:", orders.count(), "rows")
print("temp view still:", spark.sql("SELECT COUNT(*) c FROM orders").first().c, "rows")

orders.createOrReplaceTempView("orders")                # the fix: re-register
print("after re-register:", spark.sql("SELECT COUNT(*) c FROM orders").first().c, "rows")

In [ ]:
# restore the full view for the rest of the notebook
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA).csv(f"{DATA_DIR}/orders.csv")
)
orders.createOrReplaceTempView("orders")

## Parameterized SQL (Spark 3.4+) — not f-strings

In [ ]:
spark.sql(
    "SELECT order_id, product, quantity FROM orders "
    "WHERE quantity >= :min_qty AND country = :ctry",
    args={"min_qty": 2, "ctry": "IN"},
).show()

## Global temp views — note the mandatory namespace

In [ ]:
orders.createOrReplaceGlobalTempView("orders_g")
spark.sql("SELECT COUNT(*) AS n FROM global_temp.orders_g").show()

# visible from a DIFFERENT session in the same application:
spark2 = spark.newSession()
print("other session sees global:", spark2.sql("SELECT COUNT(*) AS n FROM global_temp.orders_g").first().n)
try:
    spark2.sql("SELECT COUNT(*) FROM orders")   # plain temp view: session-scoped
except Exception as e:
    print("other session does NOT see temp view:", str(e).splitlines()[0])

## Exercises

1. Rewrite lesson 3.4's category summary (orders, revenue, avg price, distinct customers per category) as one SQL query. Compare `.explain()` output with the DataFrame version — same plan shape?
2. Reproduce the 3.5 three-valued-logic demo in SQL: `WHERE country != 'IN'` vs `WHERE country != 'IN' OR country IS NULL` — row counts?
3. Parameterize exercise 2's threshold query over a list of countries with `IN (:c1, :c2)`-style args — then try passing the whole list as ONE parameter. What happens?
4. Drop both views (`spark.catalog.dropTempView`, `dropGlobalTempView`) and verify with a failing query.

In [ ]:
# your exercise code here
